## Importing Necessary Libraries

In [7]:
import pandas as pd
import numpy as np
import os
import glob
import sys
from pathlib import Path
from dotenv import load_dotenv

from sklearn.naive_bayes import GaussianNB

from nltk.stem import PorterStemmer
from nltk.tokenize import SpaceTokenizer

from pyspark.sql import SparkSession, functions as F

In [4]:
# Hardcoding since working with venv
load_dotenv()

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PATH'] = os.environ.get('HADOOP_HOME', '') + r'\bin;' + os.environ.get('PATH', '')

print(f"JAVA_HOME: {os.environ.get('JAVA_HOME')}")
print(f"HADOOP_HOME: {os.environ.get('HADOOP_HOME')}")
print(f"PYSPARK_PYTHON: {os.environ.get('PYSPARK_PYTHON')}")
print(f"PYSPARK_DRIVER_PYTHON: {os.environ.get('PYSPARK_DRIVER_PYTHON')}")
print(f"PATH: {os.environ.get('PATH')}")

JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-11.0.29.7-hotspot
HADOOP_HOME: C:\hadoop
PYSPARK_PYTHON: c:\Users\User\miniforge3\envs\test\python.exe
PYSPARK_DRIVER_PYTHON: c:\Users\User\miniforge3\envs\test\python.exe
PATH: C:\hadoop\bin;C:\hadoop\bin;c:\Users\User\miniforge3\envs\test;C:\Users\User\miniforge3\envs\test;C:\Users\User\miniforge3\envs\test\Library\mingw-w64\bin;C:\Users\User\miniforge3\envs\test\Library\usr\bin;C:\Users\User\miniforge3\envs\test\Library\bin;C:\Users\User\miniforge3\envs\test\Scripts;C:\Users\User\miniforge3\envs\test\bin;C:\Users\User\miniforge3\condabin;C:\Users\User\AppData\Local\Programs\Microsoft VS Code;C:\Program Files\Eclipse Adoptium\jdk-11.0.29.7-hotspot\bin;C:\Program Files\Common Files\Oracle\Java\javapath;C:\Program Files (x86)\Common Files\Oracle\Java\java8path;C:\Program Files (x86)\Common Files\Oracle\Java\javapath;C:\Windows\system32;C:\Windows;C:\Windows\System32\Wbem;C:\Windows\System32\WindowsPowerShell\v1.0;C:\Windows\System32\Open

## Loading Spark Context

In [2]:
spark = SparkSession.builder \
    .appName("SEEP") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

sc = spark.sparkContext

print(f"Spark version: {spark.version}")
print(f"App name: {sc.appName}")
print(f"Master: {sc.master}")

Spark version: 3.5.8
App name: SEEP
Master: local[*]


In [ ]:
# spark.stop()

## Loading Data

In [14]:
pattern_reviews = r'../data/reviews/reviews/steam_reviews_*'

files_reviews = glob.glob(pattern_reviews)
files_reviews[:5]

['../data/reviews/reviews\\steam_reviews_2025-08-22_0.parquet',
 '../data/reviews/reviews\\steam_reviews_2025-08-22_1.parquet',
 '../data/reviews/reviews\\steam_reviews_2025-08-22_10.parquet',
 '../data/reviews/reviews\\steam_reviews_2025-08-22_100.parquet',
 '../data/reviews/reviews\\steam_reviews_2025-08-22_101.parquet']

In [15]:
pattern_games = r'../data/games/games/steam_games_*'

files_games = glob.glob(pattern_games)
files_games[:5]

['../data/games/games\\steam_games_2025-08-12_0.parquet',
 '../data/games/games\\steam_games_2025-08-12_1.parquet',
 '../data/games/games\\steam_games_2025-08-12_10.parquet',
 '../data/games/games\\steam_games_2025-08-12_11.parquet',
 '../data/games/games\\steam_games_2025-08-12_12.parquet']

In [16]:
# check if all parquet in reviews
all_file_types_review = set()
for file in files_reviews:
    all_file_types_review.add(file.split('.')[-1])

all_file_types_review

{'parquet'}

In [17]:
# check if all parquet in reviews
all_file_types_games = set()
for file in files_games:
    all_file_types_games.add(file.split('.')[-1])

all_file_types_games

{'parquet'}

In [21]:
df_reviews = spark.read.parquet(*files_reviews)
df_reviews.show(5)

+---------+-----------------+-------+----------------+-----------------------+------------------+---------------+-----------+-----------+--------+--------------------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+--------------+-----------------+---------------------------+--------------------+-----------+
|   rec_id|        author_id|  appid|playtime_forever|playtime_last_two_weeks|playtime_at_review|num_games_owned|num_reviews|last_played|language|              review|timestamp_created|timestamp_updated|voted_up|votes_up|votes_funny|weighted_vote_score|comment_count|steam_purchase|received_for_free|written_during_early_access|primarily_steam_deck|scrape_date|
+---------+-----------------+-------+----------------+-----------------------+------------------+---------------+-----------+-----------+--------+--------------------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+---------

In [22]:
df_games = spark.read.parquet(*files_games)
df_games.show(5)

+-------+--------------------+----+------------+-------+-----------------------+---------------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+---------------+-----------+-------------+------------+-----------+---------------+---+------------+--------------------+-----------+
|  appid|                name|type|required_age|is_free|minimum_pc_requirements|recommended_pc_requirements|controller_support|detailed_description|      about_the_game|   short_description| supported_languages|        header_image|          developers|          publishers| price|          categories|              genres|windows_support|mac_support|linux_support|release_date|coming_soon|recommendations|dlc|review_score|   review_score_desc|scrape_date|
+-------+--------------------+----+------------+-------+-----------------------+------

In [26]:
df_games_id = spark.read.parquet('../data/games/games/steam_ids.parquet')
df_games_id.show()

+-------+---------------------------+
|  appid|                       name|
+-------+---------------------------+
|3058980|       Veilcrest: Trial ...|
| 857060|                CCCP CALLS!|
| 477270|              Lightblade VR|
| 626522|       Fate/EXTELLA - Or...|
|1310390|           Mine Trap Reborn|
| 852370|       Fantasy Grounds -...|
|2143600|             Axom: Conquest|
|2490800|閃攻機人アスラ - ASURA T...|
|3772830|       Haunted Room : 20...|
|2512560|            Butcher's Creek|
|1700505|       Tiger Fighter 193...|
|2497080|              Roody:2d Demo|
| 791670|       Chess of Blades -...|
|2674800|       CUSTOM ORDER MAID...|
|2420300|                Rap Attack!|
|1822510|               Fog & Silver|
| 669630|                   Hollowed|
|1492370|            Retis Tormentum|
|3107930|       Memology 2: old t...|
|2267180|       The Ranch of Rive...|
+-------+---------------------------+
only showing top 20 rows



## Data Preprocessing and Initial Exploration

#### For Reviews

In [23]:
# No. of records
df_reviews.count()

91013016

In [24]:
# Checking for null values
for c in df_reviews.columns:
    null_count = df_reviews.filter(F.col(c).isNull()).count()
    print(f"{c}: {null_count}")

rec_id: 0
author_id: 0
appid: 0
playtime_forever: 53979
playtime_last_two_weeks: 83035153
playtime_at_review: 119223
num_games_owned: 45257560
num_reviews: 0
last_played: 53979
language: 0
review: 0
timestamp_created: 0
timestamp_updated: 0
voted_up: 0
votes_up: 0
votes_funny: 0
weighted_vote_score: 0
comment_count: 0
steam_purchase: 0
received_for_free: 0
written_during_early_access: 0
primarily_steam_deck: 0
scrape_date: 0


#### Removing high null value columns 

In [30]:
df_col_rem = df_reviews.drop('playtime_last_two_weeks', 'num_games_owned')
df_col_rem.show(5)

+---------+-----------------+-------+----------------+------------------+-----------+-----------+--------+--------------------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+--------------+-----------------+---------------------------+--------------------+-----------+
|   rec_id|        author_id|  appid|playtime_forever|playtime_at_review|num_reviews|last_played|language|              review|timestamp_created|timestamp_updated|voted_up|votes_up|votes_funny|weighted_vote_score|comment_count|steam_purchase|received_for_free|written_during_early_access|primarily_steam_deck|scrape_date|
+---------+-----------------+-------+----------------+------------------+-----------+-----------+--------+--------------------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+--------------+-----------------+---------------------------+--------------------+-----------+
|160867953|76561198219856347|11051

In [32]:
df_col_rem.dtypes

[('rec_id', 'bigint'),
 ('author_id', 'bigint'),
 ('appid', 'bigint'),
 ('playtime_forever', 'bigint'),
 ('playtime_at_review', 'bigint'),
 ('num_reviews', 'bigint'),
 ('last_played', 'bigint'),
 ('language', 'string'),
 ('review', 'string'),
 ('timestamp_created', 'bigint'),
 ('timestamp_updated', 'bigint'),
 ('voted_up', 'boolean'),
 ('votes_up', 'bigint'),
 ('votes_funny', 'bigint'),
 ('weighted_vote_score', 'double'),
 ('comment_count', 'bigint'),
 ('steam_purchase', 'boolean'),
 ('received_for_free', 'boolean'),
 ('written_during_early_access', 'boolean'),
 ('primarily_steam_deck', 'boolean'),
 ('scrape_date', 'date')]

In [36]:
df_w_names = df_col_rem.join(df_games_id, 'appid')
df_w_names.show(5)

+-------+---------+-----------------+----------------+------------------+-----------+-----------+--------+--------------------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+--------------+-----------------+---------------------------+--------------------+-----------+-----------------+
|  appid|   rec_id|        author_id|playtime_forever|playtime_at_review|num_reviews|last_played|language|              review|timestamp_created|timestamp_updated|voted_up|votes_up|votes_funny|weighted_vote_score|comment_count|steam_purchase|received_for_free|written_during_early_access|primarily_steam_deck|scrape_date|             name|
+-------+---------+-----------------+----------------+------------------+-----------+-----------+--------+--------------------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+--------------+-----------------+---------------------------+--------------------+-----------

In [45]:
df_w_names.groupBy('name').agg(F.round(F.avg('num_reviews'), 2).alias('Average No of Reviews'),
                               F.round(F.avg('playtime_forever'), 2).alias('Average Overall Playtime'))\
                               .sort("Average No of Reviews", ascending = False).show(truncate = False)

+---------------------------------------+---------------------+------------------------+
|name                                   |Average No of Reviews|Average Overall Playtime|
+---------------------------------------+---------------------+------------------------+
|Mystery Solitaire. The Black Raven 5   |18183.0              |269.0                   |
|For Food Sake! VR                      |18166.0              |283.0                   |
|Jigsaw Puzzle Womens Day               |18166.0              |281.0                   |
|1001 Jigsaw Detective 3                |18159.0              |250.0                   |
|Astra Wing                             |18155.0              |406.0                   |
|Unlucky Sniper                         |18149.0              |266.0                   |
|Mystery Solitaire. The Black Raven 6   |18149.0              |249.0                   |
|Jigsaw Boom 2                          |18149.0              |261.0                   |
|Blue Time           

#### For games

In [25]:
# No. of records
df_games.count()

196310

In [27]:
# Checking for null values
for c in df_games.columns:
    null_count = df_games.filter(F.col(c).isNull()).count()
    print(f"{c}: {null_count}")

appid: 0
name: 0
type: 0
required_age: 1
is_free: 0
minimum_pc_requirements: 27
recommended_pc_requirements: 68552
controller_support: 144049
detailed_description: 0
about_the_game: 0
short_description: 0
supported_languages: 0
header_image: 0
developers: 0
publishers: 0
price: 65825
categories: 0
genres: 0
windows_support: 0
mac_support: 0
linux_support: 0
release_date: 0
coming_soon: 0
recommendations: 174184
dlc: 0
review_score: 11
review_score_desc: 11
scrape_date: 0


In [34]:
df_games.show(5)

+-------+--------------------+----+------------+-------+-----------------------+---------------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+---------------+-----------+-------------+------------+-----------+---------------+---+------------+--------------------+-----------+
|  appid|                name|type|required_age|is_free|minimum_pc_requirements|recommended_pc_requirements|controller_support|detailed_description|      about_the_game|   short_description| supported_languages|        header_image|          developers|          publishers| price|          categories|              genres|windows_support|mac_support|linux_support|release_date|coming_soon|recommendations|dlc|review_score|   review_score_desc|scrape_date|
+-------+--------------------+----+------------+-------+-----------------------+------